## 이 노트북의 흐름 (2주차 과제피드백 체크리스트 대응)

과제피드백 체크리스트를 하나씩 확인하며 진행한 인과 사슬입니다. 왼쪽 사이드바의 **목차(개요, Table of contents)** 아이콘을 누르면 아래 각 단계의 `###` 제목이 자동으로 나열되니, 원하는 단계로 바로 이동할 수 있습니다.

1. **YOLO26 학습(train)** — baseline 학습(`epochs=10`) + 학습 곡선/과적합 확인
2. **TP / FN / FP 시각적 확인** — IoU 기반 자동 채점 도구 작성, FP(중복 예측)와 FN(학습 부족)이 서로 다른 원인임을 확인
3. **하이퍼파라미터 실험** — 2번에서 찾은 원인을 각각 겨냥: 실험1(epochs↑, FN 해결) / 실험2(conf↑, FP 해결). `fraction=0.1` 실패 사례도 원인과 함께 남겨둠
4. **train/val/test 이미지 중복(leakage) 확인** — 3번까지의 완벽에 가까운 수치를 그대로 믿지 않고 별도 검증 (결과: leakage 없음)
5. **학습에 안 쓰인 새 이미지로 추론 테스트** — synthetic에서 완벽했던 모델이 실제 사진에서 무너짐을 발견, 해상도 문제 vs 각도 문제를 분리 진단
6. **실험 3: Perspective/Affine Augmentation** — 5번에서 찾은 각도 domain gap을 완화하기 위한 시도 (완전 해결은 아님, 부분 개선)
7. **추론 속도 / 모델 크기 측정 + 최종 모델 선정** — baseline/실험1/실험3을 속도·크기·정확도·일반화 기준으로 종합 비교, 최종 모델 결정

(데이터 분포 사전 분석은 `01_데이터셋_구축_코드.ipynb`에서 진행했습니다.)

### colab runtime 변경
- 코랩 런타임을 T4 GPU로 변경해주세요.
- 무료 버전 코랩을 사용할 경우, 하루 할당된 GPU가 제한되어 있어서 학습을 진행하다가 런타임이 끊길 수 있습니다.
- 로컬에 파이썬 설치가 가능하다면 로컬에서 진행하는 것을 추천드립니다. (GPU가 있는 경우!)
- 로컬에서 진행할 경우엔 conda를 활용하여 가상환경을 세팅해주세요.

### mount google drive
- 코랩을 활용할 경우, 구글 드라이브에 있는 데이터를 사용하기 위해서 마운트 작업이 필요합니다.
- 로컬에서 진행할 경우엔 생략해주세요.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

### Import Libraray

In [ ]:
# pip install을 통해 ultralytics(YOLO26) 패키지 설치
!pip install ultralytics

In [ ]:
import os
import cv2
import glob
import shutil
import warnings
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

In [ ]:
# matplotlib 그래프에서 한글이 깨지지 않도록 한글 폰트 설치 및 적용
!apt-get -qq install fonts-nanum > /dev/null
import matplotlib.font_manager as fm

fontpath = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(fontpath)  # 폰트 매니저 캐시에 직접 등록 (이게 빠지면 이름만 지정되고 실제로는 못 찾음)

font_name = fm.FontProperties(fname=fontpath).get_name()
plt.rc('font', family=font_name)
plt.rcParams['axes.unicode_minus'] = False  # 한글 폰트 사용 시 마이너스(-) 기호 깨짐 방지

In [ ]:
warnings.filterwarnings(action='ignore')

In [ ]:
# 현재 실행 파일 경로 확인
os.getcwd()

'/content'

### YOLO26 모델 설치 및 환경 설정

- 모델을 설치할 경로에 cd 명령어를 활용하여 들어가주세요

In [ ]:
cd /content/drive/MyDrive/comento/1

/content/drive/MyDrive/comento/1


In [ ]:
# 현재 파일 경로 확인 (원하는 경로에 들어와있는지 확인)
os.getcwd()

### YOLO26 학습(train)
- 학습을 진행할 때 사용되는 batch와 epoch에 대해서 정의도 같이 공부하면 좋을 것 같습니다.
- 학습 후에 나오는 최종 모델 파일은 `project/name/weights/best.pt` 경로에 저장됩니다.

In [ ]:
model = YOLO('yolo26n.pt')  #   YOLO26n 모델 불러오기

In [ ]:
import timeit
start = timeit.default_timer()

train_results = model.train(data='datasets_yolo/dataset.yaml', epochs=10, imgsz=224, batch=20, name='train_result', project='train_results')

end = timeit.default_timer()
print(end-start)

### 학습 곡선 확인 (loss / precision / recall / mAP) + 과적합 여부
- 2주차 과제 피드백 반영: ultralytics는 학습하는 동안 epoch별 loss/precision/recall/mAP를 자동으로 `results.png`, `results.csv`에 저장합니다.
- train loss는 계속 떨어지는데 val loss는 어느 순간부터 다시 오르면 과적합(overfitting) 신호입니다.

In [ ]:
from PIL import Image
import pandas as pd

# ultralytics가 학습 중 자동으로 저장한 epoch별 loss/precision/recall/mAP 그래프
results_img_path = 'runs/detect/train_results/train_result/results.png'
plt.figure(figsize=(14, 10))
plt.imshow(Image.open(results_img_path))
plt.axis('off')
plt.title("학습 곡선 (train/val loss, precision, recall, mAP)")
plt.show()

# 수치로도 확인 (마지막 몇 epoch의 train/val loss, mAP 추이)
results_csv_path = 'runs/detect/train_results/train_result/results.csv'
df = pd.read_csv(results_csv_path)
df.columns = df.columns.str.strip()  # 컬럼명에 공백이 섞여있는 경우 대비

print("마지막 3개 epoch 결과:")
print(df[['epoch', 'train/box_loss', 'val/box_loss', 'metrics/mAP50(B)']].tail(3))

### 전이학습 결과 확인
- 1주차 과제에서 생성한 testdataset을 기반으로 모델 성능을 평가합니다.

In [ ]:
# YOLO26 모델 성능 평가(val)
val_results = model.val(data='datasets_yolo/dataset.yaml', imgsz=224, batch=60, name='val_result', project='val_results')

### 추론(inference) 진행
- 학습에 사용되지 않은 이미지를 한 장씩 넣어서 추론 결과 확인해보기
- 추론 속도와 추론 결과 이미지 확인

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
trained_model = YOLO('runs/detect/train_results/train_result/weights/best.pt')  # baseline 모델(전체 데이터로 학습) 불러오기

In [ ]:
# 추론을 진행할 이미지 불러오기
test_img_path ='datasets_yolo/images/test/syn_01921.png'
img = Image.open(test_img_path)

# YOLO26 모델을 통해 추론 진행
test_results = trained_model.predict(source=img, save=True, conf=0.25, imgsz=224, name='test_result', project='test_results')

In [ ]:
# 추론 결과 확인하기 (탐지된 차량 개수 + 결과 이미지)
result = test_results[0]
print(len(result))
# Save or display results
result.show()
plt.imshow(result.plot())
plt.axis('off')
plt.show()

### TP / FN / FP 시각적 확인
- 2주차 과제 피드백 반영: test 이미지 여러 장에 대해 예측 박스와 정답(GT) 박스를 함께 그려서 TP/FP/FN 사례를 눈으로 확인합니다.
- TP: 예측 박스가 정답 박스와 겹치는 경우 / FP: 정답이 없는데 예측한 경우 / FN: 정답인데 예측을 못한 경우

In [ ]:
# test 이미지 전체 개수 확인 후 frac 비율만큼 확인할 샘플 개수 계산
test_image_dir = 'datasets_yolo/images/test'
test_images = glob.glob(f'{test_image_dir}/*.png')
print(f"test 이미지 전체 개수: {len(test_images)}")

frac = 0.1  # test 이미지 중 10%만 뽑아서 TP/FP/FN 눈으로 확인
sample_n = max(1, int(len(test_images) * frac))
print(f"확인할 샘플 이미지 개수: {sample_n}")



In [ ]:
def draw_gt_vs_pred(image_path: str, annotation_path: str, result):
    ''' 정답(GT, 파랑) 박스와 예측(빨강) 박스를 한 이미지 위에 함께 그리는 함수
    Args:
        image_path (str): 이미지 파일 경로
        annotation_path (str): 정답 라벨 파일 경로 (.txt, YOLO 형식)
        result: YOLO predict() 결과 객체 (result.boxes.xyxy에 예측 박스 픽셀 좌표가 담겨 있음)
    '''
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_height, img_width = image.shape[:2]

    # 정답 박스 그리기 (파랑)
    with open(annotation_path, 'r') as f:
        lines = f.readlines()
    for line in lines:
        values = list(map(float, line.strip().split(' ')))
        x_center, y_center, box_width, box_height = values[1:5]
        x_min = int(round((x_center - box_width / 2) * img_width))
        y_min = int(round((y_center - box_height / 2) * img_height))
        x_max = int(round((x_center + box_width / 2) * img_width))
        y_max = int(round((y_center + box_height / 2) * img_height))
        cv2.rectangle(image, (x_min, y_min), (x_max, y_max), (0, 0, 255), 2)

    # 예측 박스 그리기 (빨강)
    pred_boxes = result.boxes.xyxy.cpu().numpy()
    for x_min, y_min, x_max, y_max in pred_boxes:
        cv2.rectangle(image, (int(x_min), int(y_min)), (int(x_max), int(y_max)), (255, 0, 0), 2)

    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"{os.path.basename(image_path)}  (파랑=정답 GT, 빨강=예측 Pred)")
    plt.show()

In [ ]:
def calculate_iou(box1, box2):
    # box1, box2: (x_min, y_min, x_max, y_max) 픽셀 좌표
    x_min1, y_min1, x_max1, y_max1 = box1
    x_min2, y_min2, x_max2, y_max2 = box2

    inter_x_min = max(x_min1, x_min2)
    inter_y_min = max(y_min1, y_min2)
    inter_x_max = min(x_max1, x_max2)
    inter_y_max = min(y_max1, y_max2)

    inter_width = max(0, inter_x_max - inter_x_min)
    inter_height = max(0, inter_y_max - inter_y_min)
    inter_area = inter_width * inter_height

    area1 = (x_max1 - x_min1) * (y_max1 - y_min1)
    area2 = (x_max2 - x_min2) * (y_max2 - y_min2)
    union_area = area1 + area2 - inter_area

    if union_area == 0:
        return 0
    return inter_area / union_area


def match_gt_pred(gt_boxes, pred_boxes, iou_threshold=0.5):
    ''' IoU 기준으로 정답 박스와 예측 박스를 매칭해서 TP/FP/FN 개수를 셉니다.
    Args:
        gt_boxes (list): 정답 박스 리스트, 각 원소는 (x_min, y_min, x_max, y_max) 픽셀 좌표
        pred_boxes (list): 예측 박스 리스트, 같은 형식
        iou_threshold (float): 이 값 이상이면 같은 차로 매칭 (mAP50과 동일하게 기본값 0.5)
    Returns:
        tp, fp, fn (int, int, int)
    '''
    # 가능한 모든 (정답, 예측) 쌍의 IoU를 구해서, 임계값을 넘는 쌍만 후보로 남김
    candidate_pairs = []
    for gi, gt_box in enumerate(gt_boxes):
        for pi, pred_box in enumerate(pred_boxes):
            iou = calculate_iou(gt_box, pred_box)
            if iou >= iou_threshold:
                candidate_pairs.append((iou, gi, pi))

    # IoU가 높은 쌍부터 먼저 매칭 (그리디 매칭)
    candidate_pairs.sort(reverse=True)

    matched_gt = set()
    matched_pred = set()
    for iou, gi, pi in candidate_pairs:
        if gi not in matched_gt and pi not in matched_pred:
            matched_gt.add(gi)
            matched_pred.add(pi)

    tp = len(matched_gt)
    fn = len(gt_boxes) - len(matched_gt)
    fp = len(pred_boxes) - len(matched_pred)
    return tp, fp, fn

In [ ]:
import random

# test 이미지 중 sample_n장을 랜덤 샘플링해서 정답 vs 예측 박스 비교 + IoU 기반 TP/FP/FN 집계
test_label_dir = 'datasets_yolo/labels/test'
random.seed(42)  # 세션이 바뀌어도 항상 같은 샘플이 뽑히도록 고정 (exp들끼리 공정한 비교를 위해)
sample_test_images = random.sample(test_images, sample_n)

IOU_THRESHOLD = 0.5  # mAP50과 동일한 기준 (IoU 0.5 이상이면 같은 차로 매칭)

total_tp, total_fp, total_fn = 0, 0, 0

for img_path in sample_test_images:
    annotation_path = os.path.join(test_label_dir, os.path.basename(img_path).replace('.png', '.txt'))
    pred_result = trained_model.predict(source=img_path, conf=0.25, imgsz=224, verbose=False)[0]

    img_height, img_width = cv2.imread(img_path).shape[:2]

    # 정답 박스를 픽셀 좌표로 변환
    with open(annotation_path, 'r') as f:
        gt_lines = f.readlines()
    gt_boxes = []
    for line in gt_lines:
        values = list(map(float, line.strip().split(' ')))
        x_center, y_center, box_width, box_height = values[1:5]
        x_min = (x_center - box_width / 2) * img_width
        y_min = (y_center - box_height / 2) * img_height
        x_max = (x_center + box_width / 2) * img_width
        y_max = (y_center + box_height / 2) * img_height
        gt_boxes.append((x_min, y_min, x_max, y_max))

    pred_boxes = [tuple(box) for box in pred_result.boxes.xyxy.cpu().numpy()]

    tp, fp, fn = match_gt_pred(gt_boxes, pred_boxes, iou_threshold=IOU_THRESHOLD)
    total_tp += tp
    total_fp += fp
    total_fn += fn
    print(f"{os.path.basename(img_path)} - TP: {tp}, FP: {fp}, FN: {fn}")

    draw_gt_vs_pred(img_path, annotation_path, pred_result)

print(f"\n샘플 {sample_n}장 합계 - TP: {total_tp}, FP: {total_fp}, FN: {total_fn}")
precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
print(f"Precision: {precision:.3f}, Recall: {recall:.3f}")

### 하이퍼파라미터 실험
- 위에서 확인한 `runs/detect/train_results/train_result` = baseline(전체 데이터), `train_result-2` = fraction=0.1 실험 모델
- **주의**: `train_result-2`(fraction=0.1)는 TP=0으로 완전히 실패했습니다. 데이터를 10%로 줄이면서 epoch는 그대로 둬서, 1 epoch당 가중치 업데이트 횟수(= 데이터 개수 / batch 크기)가 1/10로 줄어 학습이 거의 안 된 것으로 보입니다. 그래서 이후 실험은 `fraction`을 빼고(=1.0, 전체 데이터) 진행합니다.
- 여러 모델(baseline, 실험1, 실험2 ...)을 같은 기준으로 비교하기 위해 평가 로직을 함수로 정리합니다.
- 새 실험을 돌릴 때는 `name`을 실험마다 다르게 줘서 (`exp1_epochs` 등) 결과 폴더가 헷갈리지 않도록 합니다.

In [ ]:
def evaluate_model_on_samples(weights_path, sample_images, label_dir, conf=0.25, imgsz=224, iou_threshold=0.5, nms_iou=0.7):
    ''' 주어진 가중치로 sample_images에 대해 TP/FP/FN을 집계하고 Precision/Recall을 반환
    Args:
        weights_path (str): 평가할 모델의 best.pt 경로
        sample_images (list): 평가에 사용할 이미지 경로 리스트
        label_dir (str): 정답 라벨(.txt) 폴더 경로
        conf, imgsz (float, int): predict()에 사용할 설정
        iou_threshold (float): TP 판정 IoU 임계값
        nms_iou (float): 중복 박스 억제(NMS) 임계값. 낮을수록 중복 박스를 더 적극적으로 제거함 (ultralytics 기본값 0.7)
    Returns:
        dict: weights_path, tp, fp, fn, precision, recall
    '''
    eval_model = YOLO(weights_path)
    total_tp, total_fp, total_fn = 0, 0, 0

    for img_path in sample_images:
        annotation_path = os.path.join(label_dir, os.path.basename(img_path).replace('.png', '.txt'))
        pred_result = eval_model.predict(source=img_path, conf=conf, imgsz=imgsz, iou=nms_iou, verbose=False)[0]

        img_height, img_width = cv2.imread(img_path).shape[:2]
        with open(annotation_path, 'r') as f:
            gt_lines = f.readlines()
        gt_boxes = []
        for line in gt_lines:
            values = list(map(float, line.strip().split(' ')))
            x_center, y_center, box_width, box_height = values[1:5]
            x_min = (x_center - box_width / 2) * img_width
            y_min = (y_center - box_height / 2) * img_height
            x_max = (x_center + box_width / 2) * img_width
            y_max = (y_center + box_height / 2) * img_height
            gt_boxes.append((x_min, y_min, x_max, y_max))

        pred_boxes = [tuple(box) for box in pred_result.boxes.xyxy.cpu().numpy()]

        tp, fp, fn = match_gt_pred(gt_boxes, pred_boxes, iou_threshold=iou_threshold)
        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    print(f"[{weights_path}] nms_iou={nms_iou}, conf={conf} -> TP: {total_tp}, FP: {total_fp}, FN: {total_fn}, Precision: {precision:.3f}, Recall: {recall:.3f}")
    return {'weights_path': weights_path, 'nms_iou': nms_iou, 'conf': conf, 'tp': total_tp, 'fp': total_fp, 'fn': total_fn, 'precision': precision, 'recall': recall}

In [ ]:
# baseline vs fraction=0.1 실험을 같은 sample_test_images 기준으로 비교
comparison_results = []
comparison_results.append(evaluate_model_on_samples('runs/detect/train_results/train_result/weights/best.pt', sample_test_images, test_label_dir))
comparison_results.append(evaluate_model_on_samples('runs/detect/train_results/train_result-2/weights/best.pt', sample_test_images, test_label_dir))

### 실험 1: epochs 늘리기
- baseline은 `epochs=10`이었습니다. 학습이 부족했을 가능성을 확인하기 위해 `epochs=30`으로 늘려서 재학습합니다 (`fraction`은 빼서 전체 데이터 사용).
- 결과 폴더가 baseline(`train_result`)과 겹치지 않도록 `name='exp1_epochs'`로 저장합니다.

In [ ]:
import timeit
start = timeit.default_timer()

exp1_model = YOLO('yolo26n.pt')
exp1_train_results = exp1_model.train(data='datasets_yolo/dataset.yaml', epochs=30, imgsz=224, batch=20, name='exp1_epochs', project='train_results')

end = timeit.default_timer()
print(end - start)

In [ ]:
# baseline과 exp1(epochs=30)을 같은 sample_test_images 기준으로 비교
comparison_results.append(evaluate_model_on_samples('runs/detect/train_results/exp1_epochs/weights/best.pt', sample_test_images, test_label_dir))

print("\n=== 비교 요약 ===")
for r in comparison_results:
    print(f"{r['weights_path']}: Precision={r['precision']:.3f}, Recall={r['recall']:.3f}, TP={r['tp']}, FP={r['fp']}, FN={r['fn']}")

### exp1 과적합 여부 확인
- Precision/Recall이 거의 완벽하게 나와서, cell-17과 같은 방식으로 exp1의 학습 곡선을 확인해 val loss가 끝까지 잘 떨어졌는지(=진짜 잘 배운 것) 확인합니다.

In [ ]:
from PIL import Image
import pandas as pd

# exp1(epochs=30)의 epoch별 loss/precision/recall/mAP 그래프
exp1_results_img_path = 'runs/detect/train_results/exp1_epochs/results.png'
plt.figure(figsize=(14, 10))
plt.imshow(Image.open(exp1_results_img_path))
plt.axis('off')
plt.title("exp1(epochs=30) 학습 곡선")
plt.show()

# 수치로도 확인 (마지막 몇 epoch의 train/val loss, mAP 추이)
exp1_results_csv_path = 'runs/detect/train_results/exp1_epochs/results.csv'
exp1_df = pd.read_csv(exp1_results_csv_path)
exp1_df.columns = exp1_df.columns.str.strip()

print("exp1 마지막 5개 epoch 결과:")
print(exp1_df[['epoch', 'train/box_loss', 'val/box_loss', 'metrics/mAP50(B)']].tail(5))

In [ ]:
# FP/FN이 있었던 이미지들의 GT/Pred 박스 좌표를 한 번에 확인
debug_image_names = [
    'syn_01623.png', 'syn_01788.png', 'syn_00077.png', 'syn_01907.png',
    'syn_01094.png', 'syn_00053.png', 'syn_00785.png', 'syn_00186.png', 'syn_00067.png',
]

for img_name in debug_image_names:
    img_path = os.path.join(test_image_dir, img_name)
    annotation_path = os.path.join(test_label_dir, img_name.replace('.png', '.txt'))
    result = trained_model.predict(source=img_path, conf=0.25, imgsz=224, verbose=False)[0]

    img_height, img_width = cv2.imread(img_path).shape[:2]
    print(f"\n===== {img_name} (이미지 크기: {img_width}x{img_height}) =====")

    # 정답(GT) 박스: YOLO 정규화 좌표 -> 픽셀 좌표로 변환해서 출력 (가로/세로 크기도 같이)
    with open(annotation_path, 'r') as f:
        gt_lines = f.readlines()
    print(f"정답(GT) 박스 {len(gt_lines)}개:")
    for line in gt_lines:
        values = list(map(float, line.strip().split(' ')))
        x_center, y_center, box_width, box_height = values[1:5]
        x_min = (x_center - box_width / 2) * img_width
        y_min = (y_center - box_height / 2) * img_height
        x_max = (x_center + box_width / 2) * img_width
        y_max = (y_center + box_height / 2) * img_height
        print(f"  gt: [{x_min:.1f}, {y_min:.1f}, {x_max:.1f}, {y_max:.1f}]  (가로{x_max - x_min:.0f} x 세로{y_max - y_min:.0f})")

    # 예측(Pred) 박스: 좌표 + confidence
    pred_boxes = result.boxes.xyxy.cpu().numpy()
    pred_confs = result.boxes.conf.cpu().numpy()
    print(f"예측(Pred) 박스 {len(pred_boxes)}개:")
    for i, (box, conf) in enumerate(zip(pred_boxes, pred_confs)):
        print(f"  pred {i}: {box}, conf={conf:.3f}")

In [ ]:
# exp1(epochs=30)의 남은 FP 1개가 어느 이미지인지 찾기
exp1_model_eval = YOLO('runs/detect/train_results/exp1_epochs/weights/best.pt')

for img_path in sample_test_images:
    annotation_path = os.path.join(test_label_dir, os.path.basename(img_path).replace('.png', '.txt'))
    pred_result = exp1_model_eval.predict(source=img_path, conf=0.25, imgsz=224, verbose=False)[0]

    img_height, img_width = cv2.imread(img_path).shape[:2]
    with open(annotation_path, 'r') as f:
        gt_lines = f.readlines()
    gt_boxes = []
    for line in gt_lines:
        values = list(map(float, line.strip().split(' ')))
        x_center, y_center, box_width, box_height = values[1:5]
        x_min = (x_center - box_width / 2) * img_width
        y_min = (y_center - box_height / 2) * img_height
        x_max = (x_center + box_width / 2) * img_width
        y_max = (y_center + box_height / 2) * img_height
        gt_boxes.append((x_min, y_min, x_max, y_max))

    pred_boxes = [tuple(box) for box in pred_result.boxes.xyxy.cpu().numpy()]
    tp, fp, fn = match_gt_pred(gt_boxes, pred_boxes, iou_threshold=IOU_THRESHOLD)

    if fp > 0 or fn > 0:
        print(f"{os.path.basename(img_path)} - TP: {tp}, FP: {fp}, FN: {fn}")
        for i, (box, conf) in enumerate(zip(pred_result.boxes.xyxy.cpu().numpy(), pred_result.boxes.conf.cpu().numpy())):
            print(f"  pred {i}: {box}, conf={conf:.3f}")

### 실험 2: NMS 임계값만 조정 (epochs=10 그대로, 재학습 없음)
- FP의 대부분이 "중복 예측"이었다는 걸 확인했으니, epoch를 늘리지 않고 baseline(epochs=10) 가중치 그대로 NMS `iou`만 0.7 → 0.5로 낮춰서 중복이 줄어드는지 확인합니다.
- 재학습이 필요 없는 실험이라(추론 파라미터만 변경) 빠르게 결과를 볼 수 있습니다.

In [ ]:
# baseline(epochs=10) 가중치를 그대로 쓰되, NMS iou만 0.5로 낮춰서 비교
nms_comparison_results = []
nms_comparison_results.append(evaluate_model_on_samples('runs/detect/train_results/train_result/weights/best.pt', sample_test_images, test_label_dir, nms_iou=0.7))  # 기존(기본값)
nms_comparison_results.append(evaluate_model_on_samples('runs/detect/train_results/train_result/weights/best.pt', sample_test_images, test_label_dir, nms_iou=0.5))  # NMS만 강화

print("\n=== epochs=10 그대로, NMS iou만 조정한 비교 ===")
for r in nms_comparison_results:
    print(f"nms_iou={r['nms_iou']}: Precision={r['precision']:.3f}, Recall={r['recall']:.3f}, TP={r['tp']}, FP={r['fp']}, FN={r['fn']}")

In [ ]:
# 두 해결책을 합쳐보기: epochs=30(exp1) 모델에 NMS iou=0.5까지 적용
combo_result_default = evaluate_model_on_samples('runs/detect/train_results/exp1_epochs/weights/best.pt', sample_test_images, test_label_dir, nms_iou=0.7)  # exp1 기존
combo_result_nms = evaluate_model_on_samples('runs/detect/train_results/exp1_epochs/weights/best.pt', sample_test_images, test_label_dir, nms_iou=0.5)  # exp1 + NMS 강화

print("\n=== epochs=30 + NMS 조정 조합 ===")
print(f"exp1(nms_iou=0.7, 기존): Precision={combo_result_default['precision']:.3f}, Recall={combo_result_default['recall']:.3f}, TP={combo_result_default['tp']}, FP={combo_result_default['fp']}, FN={combo_result_default['fn']}")
print(f"exp1(nms_iou=0.5, 조합): Precision={combo_result_nms['precision']:.3f}, Recall={combo_result_nms['recall']:.3f}, TP={combo_result_nms['tp']}, FP={combo_result_nms['fp']}, FN={combo_result_nms['fn']}")

In [ ]:
# syn_00067.png 한 장으로 iou 파라미터가 실제로 예측 박스 개수를 바꾸는지 직접 확인
debug_img_path = os.path.join(test_image_dir, 'syn_00067.png')

result_iou07 = trained_model.predict(source=debug_img_path, conf=0.25, imgsz=224, iou=0.7, verbose=False)[0]
result_iou05 = trained_model.predict(source=debug_img_path, conf=0.25, imgsz=224, iou=0.5, verbose=False)[0]
result_iou01 = trained_model.predict(source=debug_img_path, conf=0.25, imgsz=224, iou=0.1, verbose=False)[0]

print(f"iou=0.7: 예측 박스 {len(result_iou07.boxes)}개")
print(f"iou=0.5: 예측 박스 {len(result_iou05.boxes)}개")
print(f"iou=0.1: 예측 박스 {len(result_iou01.boxes)}개 (극단적으로 낮춰도 안 바뀌면 iou 파라미터가 이 모델엔 안 먹히는 것)")

In [ ]:
# baseline(epochs=10) 가중치 그대로, conf 값만 여러 단계로 올려서 비교
conf_comparison_results = []
for conf_value in [0.25, 0.35, 0.45, 0.55]:
    conf_comparison_results.append(evaluate_model_on_samples('runs/detect/train_results/train_result/weights/best.pt', sample_test_images, test_label_dir, conf=conf_value))

print("\n=== epochs=10 그대로, conf만 조정한 비교 ===")
for r in conf_comparison_results:
    print(f"conf={r['conf']}: Precision={r['precision']:.3f}, Recall={r['recall']:.3f}, TP={r['tp']}, FP={r['fp']}, FN={r['fn']}")

In [ ]:
# exp1(epochs=30) 가중치로, conf 값만 올려서 마지막 남은 FP(중복)까지 없앨 수 있는지 확인
exp1_conf_comparison_results = []
for conf_value in [0.25, 0.35, 0.45]:
    exp1_conf_comparison_results.append(evaluate_model_on_samples('runs/detect/train_results/exp1_epochs/weights/best.pt', sample_test_images, test_label_dir, conf=conf_value))

print("\n=== epochs=30 + conf 조정 비교 ===")
for r in exp1_conf_comparison_results:
    print(f"conf={r['conf']}: Precision={r['precision']:.3f}, Recall={r['recall']:.3f}, TP={r['tp']}, FP={r['fp']}, FN={r['fn']}")

### train/val/test 이미지 중복(leakage) 확인
- 01번 노트북의 분리 코드(`train_test_split`)는 파일 리스트를 나누는 방식이라, 구조적으로 같은 파일명이 두 세트에 동시에 들어갈 수 없습니다.
- 하지만 synthetic 데이터 특성상, **파일명은 다르지만 내용이 똑같은(또는 거의 같은) 이미지**가 서로 다른 split에 들어갔을 가능성이 있습니다. 이걸 확인하기 위해 이미지 내용 자체를 해시(MD5)로 비교합니다.

In [ ]:
import hashlib

def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

split_dirs = {
    'train': 'datasets_yolo/images/train',
    'val': 'datasets_yolo/images/val',
    'test': 'datasets_yolo/images/test',
}

hash_to_entries = {}
for split_name, dir_path in split_dirs.items():
    for img_path in tqdm(glob.glob(f'{dir_path}/*.png'), desc=split_name):
        h = file_hash(img_path)
        hash_to_entries.setdefault(h, []).append((split_name, img_path))

# 같은 해시(=완전히 같은 내용)를 가진 이미지들 중, 서로 다른 split에 걸쳐 있는 것만 leakage로 판단
cross_split_duplicates = {
    h: entries for h, entries in hash_to_entries.items()
    if len(set(split for split, _ in entries)) > 1
}

print(f"전체 이미지 개수: {sum(len(v) for v in hash_to_entries.values())}")
print(f"train/val/test에 걸쳐 중복된(leakage) 이미지 그룹 수: {len(cross_split_duplicates)}")

for h, entries in list(cross_split_duplicates.items())[:10]:
    print(entries)

### 학습에 안 쓰인 새 이미지로 추론 테스트
- 지금까지의 test 결과는 전부 synthetic 데이터(train과 같은 분포) 기준이었습니다.
- 실제 처음 보는 이미지(항공뷰/CCTV 각도의 도로·주차장 사진)를 몇 장 업로드해서, 최종 후보 모델(exp1, conf=0.45)이 실전에서도 통하는지 확인합니다.
- 정답 라벨이 없으므로 TP/FP/FN 수치 계산은 불가능하고, 결과 이미지를 눈으로 확인합니다.

In [ ]:
from google.colab import files

# 새 이미지 파일 업로드 (컴퓨터에서 파일 선택 창이 뜹니다)
uploaded = files.upload()
new_image_paths = list(uploaded.keys())
print(f"업로드된 이미지: {new_image_paths}")

In [ ]:
# 최종 후보 모델(exp1, epochs=30)로 새 이미지 추론 + 결과 시각화
final_model = YOLO('runs/detect/train_results/exp1_epochs/weights/best.pt')

for img_path in new_image_paths:
    result = final_model.predict(source=img_path, conf=0.45, imgsz=224, verbose=False)[0]
    print(f"{img_path} - 탐지된 차량 수: {len(result.boxes)}")

    plotted = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)  # result.plot()은 BGR이라 matplotlib용으로 RGB 변환
    plt.figure(figsize=(10, 10))
    plt.imshow(plotted)
    plt.axis('off')
    plt.title(img_path)
    plt.show()

In [ ]:
# imgsz를 키우면 결과가 달라지는지 확인 (각도 문제 vs 해상도 문제 구분)
for img_path in new_image_paths:
    n_224 = len(final_model.predict(source=img_path, conf=0.45, imgsz=224, verbose=False)[0].boxes)
    n_1280 = len(final_model.predict(source=img_path, conf=0.45, imgsz=1280, verbose=False)[0].boxes)
    print(f"{img_path} - imgsz=224: {n_224}개, imgsz=1280: {n_1280}개")

### 실험 3: Perspective/Affine Augmentation 추가 (각도 일반화 시도)
- 참고: 라벨 위치/시점이 특정 패턴에 몰려 있을 때, 추가 데이터 없이도 `perspective`/`shear`/`degrees` 같은 augmentation으로 각도 다양성을 인위적으로 넓힐 수 있습니다.
- ultralytics 기본값은 `perspective=0.0`, `shear=0.0`, `degrees=0.0`(전부 꺼져 있음) — 이번 실험에서 값을 넣어서 켭니다.
- epochs=30(exp1과 동일 조건)에서 augmentation만 추가해서, ①synthetic test 성능이 떨어지지 않는지 ②새 이미지(각도 다른 사진)에서 더 잘 잡는지 두 가지를 비교합니다.

In [ ]:
import timeit
start = timeit.default_timer()

exp3_model = YOLO('yolo26n.pt')
exp3_train_results = exp3_model.train(
    data='datasets_yolo/dataset.yaml', epochs=30, imgsz=224, batch=20,
    perspective=0.0005, shear=5.0, degrees=10.0,  # 각도 다양성을 위한 augmentation (기본값은 전부 0)
    name='exp3_augment', project='train_results'
)

end = timeit.default_timer()
print(end - start)

In [ ]:
# ① synthetic test에서 exp1(augmentation 없음) vs exp3(augmentation 추가) 비교
exp3_synthetic_result = evaluate_model_on_samples('runs/detect/train_results/exp3_augment/weights/best.pt', sample_test_images, test_label_dir, conf=0.45)

print("\n=== synthetic test: exp1 vs exp3 ===")
print(f"exp1 (augmentation 없음): Precision={exp1_conf_comparison_results[-1]['precision']:.3f}, Recall={exp1_conf_comparison_results[-1]['recall']:.3f}")
print(f"exp3 (augmentation 추가): Precision={exp3_synthetic_result['precision']:.3f}, Recall={exp3_synthetic_result['recall']:.3f}")

In [ ]:
# ② 새 이미지(각도가 다른 실제 사진)에서 exp1 vs exp3 비교
exp3_model_eval = YOLO('runs/detect/train_results/exp3_augment/weights/best.pt')

print(f"{'파일명':40s} {'exp1(기존)':>12s} {'exp3(augment)':>15s}")
for img_path in new_image_paths:
    n_exp1 = len(final_model.predict(source=img_path, conf=0.45, imgsz=224, verbose=False)[0].boxes)
    n_exp3 = len(exp3_model_eval.predict(source=img_path, conf=0.45, imgsz=224, verbose=False)[0].boxes)
    print(f"{img_path:40s} {n_exp1:>12d} {n_exp3:>15d}")

### 추론 속도 / 모델 크기 측정 (최종 모델 선정용)
- 2주차 과제피드백: 최종 모델은 mAP/Precision/Recall뿐 아니라 추론 속도(FPS/Latency)와 모델 크기도 같이 고려해야 합니다.
- baseline / exp1(epochs=30) / exp3(augmentation)를 같은 기준(10장 평균)으로 비교합니다.

In [ ]:
import time

def measure_inference_speed(weights_path, sample_images, imgsz=224, conf=0.45, n_warmup=2):
    ''' 평균 추론 시간(ms/image)과 모델 파일 크기(MB)를 측정
    Args:
        weights_path (str): 측정할 모델의 best.pt 경로
        sample_images (list): 측정에 사용할 이미지 경로 리스트
        imgsz, conf: predict()에 사용할 설정 (최종 채택한 값과 동일하게)
        n_warmup (int): 첫 몇 장은 GPU 초기화 등으로 느리므로 측정에서 제외
    '''
    model = YOLO(weights_path)
    for img_path in sample_images[:n_warmup]:
        model.predict(source=img_path, imgsz=imgsz, conf=conf, verbose=False)

    start = time.time()
    for img_path in sample_images:
        model.predict(source=img_path, imgsz=imgsz, conf=conf, verbose=False)
    end = time.time()

    avg_ms = (end - start) / len(sample_images) * 1000
    size_mb = os.path.getsize(weights_path) / (1024 * 1024)
    print(f"[{weights_path}] 평균 추론 시간: {avg_ms:.2f}ms/image, 모델 크기: {size_mb:.2f}MB")
    return {'weights_path': weights_path, 'avg_ms': avg_ms, 'size_mb': size_mb}


speed_size_results = []
for path in [
    'runs/detect/train_results/train_result/weights/best.pt',
    'runs/detect/train_results/exp1_epochs/weights/best.pt',
    'runs/detect/train_results/exp3_augment/weights/best.pt',
]:
    speed_size_results.append(measure_inference_speed(path, sample_test_images[:10]))

In [ ]:
# exp3만 유독 느리게 나온 게 "진짜 모델 차이"인지 "측정 순서(GPU 워밍업/자원 경쟁) 효과"인지 확인
# -> 순서를 거꾸로(exp3 먼저, baseline 마지막) 측정해서 순위가 그대로면 진짜 차이, 뒤집히면 순서 효과
speed_size_results_reversed = []
for path in [
    'runs/detect/train_results/exp3_augment/weights/best.pt',
    'runs/detect/train_results/exp1_epochs/weights/best.pt',
    'runs/detect/train_results/train_result/weights/best.pt',
]:
    speed_size_results_reversed.append(measure_inference_speed(path, sample_test_images[:10]))

### 실험 4: 학습 imgsz를 640으로 키우기 (원거리/소형 차량 인식 개선 시도)
- 배경: 3주차 FastAPI 테스트에서 가까운 차는 잡았지만 멀리 있는(화면상 작게 보이는) 차를 놓친 사례 발견. 지금까지 `imgsz`는 추론(cell-52)에서만 실험했고, 학습 자체는 계속 224로 고정돼 있었음 — 작은 물체는 224로 리사이즈되는 순간 이미 디테일이 많이 손실된 채로 학습됐을 가능성.
- exp3(augmentation) 조건은 그대로 유지(`perspective/shear/degrees`)하고 `imgsz`만 640으로 올림.
- `imgsz`를 224→640으로 올리면 계산량이 대략 (640/224)² ≈ 8배로 늘어남 → GPU 메모리 때문에 `batch`는 20→8로 낮추고, 시간 예산(1시간) 안에서 확인하기 위해 `epochs`는 우선 baseline과 같은 10으로 시작. 학습 로그에 epoch별 소요 시간이 찍히니, 10 epoch가 1시간을 넘길 것 같으면 런타임에서 중단해도 됨(그때까지의 `best.pt`는 이미 저장돼 있음).

In [ ]:
import timeit
start = timeit.default_timer()

exp4_model = YOLO('yolo26n.pt')
exp4_train_results = exp4_model.train(
    data='datasets_yolo/dataset.yaml', epochs=10, imgsz=640, batch=8,
    perspective=0.0005, shear=5.0, degrees=10.0,  # exp3와 동일한 augmentation 유지
    name='exp4_imgsz640', project='train_results'
)

end = timeit.default_timer()
print(f"소요 시간: {(end - start) / 60:.1f}분")

In [ ]:
# ① synthetic test에서 exp3(imgsz=224) vs exp4(imgsz=640) 비교
# 주의: exp4는 imgsz=640으로 학습했으니, 평가도 같은 imgsz=640으로 맞춰야 공정한 비교가 됨
exp4_synthetic_result = evaluate_model_on_samples('runs/detect/train_results/exp4_imgsz640/weights/best.pt', sample_test_images, test_label_dir, conf=0.45, imgsz=640)

print("\n=== synthetic test: exp3(224) vs exp4(640) ===")
print(f"exp3 (imgsz=224): Precision={exp3_synthetic_result['precision']:.3f}, Recall={exp3_synthetic_result['recall']:.3f}")
print(f"exp4 (imgsz=640): Precision={exp4_synthetic_result['precision']:.3f}, Recall={exp4_synthetic_result['recall']:.3f}")

# ② 새 이미지(실제 사진)에서 exp3 vs exp4 비교 (exp4는 imgsz=640으로 추론)
exp4_model_eval = YOLO('runs/detect/train_results/exp4_imgsz640/weights/best.pt')

print(f"\n{'파일명':40s} {'exp3(224)':>12s} {'exp4(640)':>12s}")
for img_path in new_image_paths:
    n_exp3 = len(exp3_model_eval.predict(source=img_path, conf=0.45, imgsz=224, verbose=False)[0].boxes)
    n_exp4 = len(exp4_model_eval.predict(source=img_path, conf=0.45, imgsz=640, verbose=False)[0].boxes)
    print(f"{img_path:40s} {n_exp3:>12d} {n_exp4:>12d}")